[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/39_ppo_loss.ipynb)

# 🔴 Hard: PPO Clipped Loss

Implement the **PPO (Proximal Policy Optimization)** **clipped surrogate loss**.

Given:
- `new_logps`: current policy log-probs $(B,)$
- `old_logps`: old policy log-probs $(B,)$
- `advantages`: advantage estimates $(B,)$

Define the ratio

$$ r_i = \exp(\text{new\_logps}_i - \text{old\_logps}_i). $$

Then compute
- $L^{\text{unclipped}}_i = r_i A_i$
- $L^{\text{clipped}}_i = \operatorname{clip}(r_i, 1-\epsilon, 1+\epsilon) A_i$

The loss is the negative batch mean of the elementwise minimum:

$$
\mathcal{L}_\text{PPO} = -\mathbb{E}_i\big[\min(L^{\text{unclipped}}_i, L^{\text{clipped}}_i)\big].
$$

Implementation notes: detach `old_logps` and `advantages` so gradients only flow through `new_logps`.

### Signature
```python
from torch import Tensor

def ppo_loss(new_logps: Tensor, old_logps: Tensor, advantages: Tensor,
             clip_ratio: float = 0.2) -> Tensor:
    """PPO clipped surrogate loss over a batch."""
```


In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.3 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn.functional as F
from torch import Tensor


In [3]:
# ✏️ YOUR IMPLEMENTATION HERE

def ppo_loss(new_logps: Tensor, old_logps: Tensor, advantages: Tensor,
             clip_ratio: float = 0.2) -> Tensor:
    diff_logps = torch.exp(new_logps - old_logps.detach())
    loss = torch.minimum(diff_logps * advantages.detach(), torch.clamp(diff_logps, min=1 - clip_ratio, max=1 + clip_ratio) * advantages.detach())
    return loss.mean()*-1.0
    pass  # -mean(min(r * adv, clamp(r, 1-clip, 1+clip) * adv)) with gradients only through new_logps


In [4]:
# 🧪 Debug
new_logps = torch.tensor([0.0, -0.2, -0.4, -0.6])
old_logps = torch.tensor([0.0, -0.1, -0.5, -0.5])
advantages = torch.tensor([1.0, -1.0, 0.5, -0.5])
print('Loss:', ppo_loss(new_logps, old_logps, advantages, clip_ratio=0.2))


Loss: tensor(-0.0488)


In [5]:
# ✅ SUBMIT
from torch_judge import check
check('ppo_loss')



🧪 Testing: PPO (Proximal Policy Optimization) Clipped Loss (Hard)
──────────────────────────────────────────────────
  ✅ [1/3] Basic shape & type (7.3ms)
  ✅ [2/3] Numeric check vs fixed value (2.6ms)
  ✅ [3/3] Gradient flows to new_logps only (12.4ms)
──────────────────────────────────────────────────
  🎉 All 3 tests passed! (22.3ms total)
  Progress saved. Run status() to see your dashboard.



In [6]:
def compute_gae(rewards, action_mask, values, gamma, lam):
    # rewards: (B, 1) scalar terminal reward   action_mask,values: (B, S)
    # last_idx = action_mask.long().cumsum(-1).argmax(-1, keepdim=True)
    # done = db,t​=1[t≥jb​]    # 1 at terminal AND after
    # scatter `rewards` into zeros(B,S) at last_idx
    # reverse loop: running (B,), next_values (B,), init 0
    # return advantages * action_mask                 # (B, S), advantages only
    ...
    last_index = torch.argmax(torch.cumsum(action_mask.long(), dim=-1), dim=-1, keepdim=True)
    reward_scattered = torch.zeros(action_mask.shape)
    reward_scattered.scatter_(-1, last_index, rewards)
    done_t = (torch.arange(action_mask.shape[1]).unsqueeze(0) >= last_index).int()
    V_next = torch.zeros(rewards.shape).squeeze()
    g = torch.zeros(rewards.shape).squeeze()
    advantage = torch.zeros(action_mask.shape)
    for t in range(action_mask.shape[1]-1, -1, -1):
      delta = reward_scattered[:,t] + gamma * (1  - done_t[:,t]) * V_next - values[:,t]
      g = delta + gamma * lam * (1  - done_t[:,t]) * g
      advantage[:,t] = g
      V_next = values[:,t]
    return advantage * action_mask


In [7]:
import torch

def _test_compute_gae():
    # ---- Test 1: shapes and dtype ----
    B, S = 2, 4
    mask = torch.tensor([[1.,1.,1.,1.],[1.,1.,0.,0.]])
    values = torch.zeros(B, S)
    rewards = torch.tensor([[1.0],[1.0]])
    A = compute_gae(rewards, mask, values, gamma=0.99, lam=0.95)
    assert A.shape == (B, S), A.shape
    assert A.dtype == torch.float32
    print("✓ 1: shape and dtype")

    # ---- Test 2: padding advantages are exactly zero ----
    assert torch.allclose(A[1, 2:], torch.zeros(2)), A[1]
    print("✓ 2: padding zeroed")

    # ---- Test 3: terminal token has no bootstrap, for ANY gamma/lam ----
    # Sentinels go in row1's REAL padding (idx 2,3); row0 is fully active so its
    # terminal value must be a real number.
    values = torch.tensor([[0.5, 1.0, 2.0, 2.0],   # row0 fully active, terminal idx3 V=2.0
                           [0.5, 1.0, 9.9, 9.9]])   # row1 padding at idx2,3 (must not leak)
    rewards = torch.tensor([[3.0],[3.0]])
    for g, l in [(1.0, 1.0), (0.99, 0.95), (0.5, 0.3)]:
        A = compute_gae(rewards, mask, values, gamma=g, lam=l)
        assert torch.allclose(A[0, 3], torch.tensor(3.0 - 2.0), atol=1e-5), (g, l, A[0, 3])
        assert torch.allclose(A[1, 1], torch.tensor(3.0 - 1.0), atol=1e-5), (g, l, A[1, 1])
    print("✓ 3: terminal drops the bootstrap, any gamma/lam; padding does not leak")

    # ---- Test 4: hand-computed full trajectory (row0, all 4 active) ----
    mask4 = torch.tensor([[1.,1.,1.,1.]])
    values4 = torch.tensor([[0.5, 1.0, 2.0, 4.0]])
    R = 3.0
    rewards4 = torch.tensor([[R]])
    A = compute_gae(rewards4, mask4, values4, gamma=1.0, lam=1.0)
    # reward after scatter = [0,0,0,3], done only at t=3, gamma=lam=1:
    # t=3: delta = 3 + 0     - 4.0 = -1.0   A3 = -1.0
    # t=2: delta = 0 + 4.0   - 2.0 =  2.0   A2 = 2.0 + A3 = 1.0
    # t=1: delta = 0 + 2.0   - 1.0 =  1.0   A1 = 1.0 + A2 = 2.0
    # t=0: delta = 0 + 1.0   - 0.5 =  0.5   A0 = 0.5 + A1 = 2.5
    expected = torch.tensor([[2.5, 2.0, 1.0, -1.0]])
    assert torch.allclose(A, expected, atol=1e-5), A
    print("✓ 4: matches hand-computed GAE")

    # ---- Test 5: gamma=lam=1 => returns collapse to the Monte-Carlo return R ----
    ret = A + values4
    assert torch.allclose(ret[mask4.bool()], torch.full((int(mask4.sum()),), R), atol=1e-5), ret
    print("✓ 5: gamma=lam=1 gives returns == total reward everywhere")

    # ---- Test 6: gamma=lam=0 => A = (scattered reward - value) on active tokens ----
    A = compute_gae(rewards4, mask4, values4, gamma=0.0, lam=0.0)
    expected = torch.tensor([[0-0.5, 0-1.0, 0-2.0, R-4.0]])
    assert torch.allclose(A, expected, atol=1e-5), A
    print("✓ 6: gamma=lam=0 reduces to reward-minus-baseline")

    print("\nAll compute_gae tests passed.")

_test_compute_gae()

✓ 1: shape and dtype
✓ 2: padding zeroed
✓ 3: terminal drops the bootstrap, any gamma/lam; padding does not leak
✓ 4: matches hand-computed GAE
✓ 5: gamma=lam=1 gives returns == total reward everywhere
✓ 6: gamma=lam=0 reduces to reward-minus-baseline

All compute_gae tests passed.


In [8]:
def apply_reward_kl(total_reward, log_probs, ref_log_probs, action_mask, beta, kl_estimator="kl3"):
    # total_reward:            (B, 1) scalar reward per sequence
    # log_probs, ref_log_probs:(B, S) per-token logps of the SAME tokens under policy vs reference
    # action_mask:             (B, S)
    # returns (B, 1):  total_reward - beta * masked_mean_over_tokens(kl)
    #
    # kl estimators (rho = log_probs - ref_log_probs, masked to 0 on padding FIRST):
    #   kl1 = -rho          kl2 = rho**2 / 2          kl3 = exp(rho) - 1 - rho
    # reduction:  masked_mean(kl, mask, dim=-1, keepdim=True)  ->  (B, 1)
    ...
    rho  = (log_probs - ref_log_probs) * action_mask
    kl1  = -rho
    kl2 = rho * rho / 2.0
    kl3 = torch.exp(rho) - 1 - rho
    if kl_estimator=="kl1":
      kl  = -rho
    elif kl_estimator=="kl2":
      kl = rho * rho / 2.0
    else:
      kl = torch.exp(rho) - 1 - rho
    n = torch.sum(action_mask, dim=-1, keepdim=True).clamp(1.0)
    kl = torch.sum(kl, dim=-1, keepdim=True) / n
    return total_reward - beta * kl



In [9]:
import torch

def _test_apply_reward_kl():
    # row0: 3 active tokens + 1 padding slot holding overflow garbage (100.0)
    # row1: fully active, zero divergence (logp == ref)
    mask  = torch.tensor([[1.,1.,1.,0.],[1.,1.,1.,1.]])
    ref   = torch.zeros(2, 4)
    logp  = torch.tensor([[0.2, -0.1, 0.5, 100.0],
                          [0.0,  0.0, 0.0,   0.0]])
    total = torch.tensor([[1.0],[2.0]])
    beta  = 0.2

    # ---- Test 1: shape and dtype (B,1) -> (B,1) ----
    out = apply_reward_kl(total, logp, ref, mask, beta, "kl3")
    assert out.shape == (2, 1), out.shape
    assert out.dtype == torch.float32
    print("✓ 1: shape and dtype")

    # ---- Test 2: beta=0 is the identity (no penalty), even with random logps ----
    out0 = apply_reward_kl(total, torch.randn(2,4), torch.randn(2,4), mask, 0.0, "kl3")
    assert torch.allclose(out0, total), out0
    print("✓ 2: beta=0 leaves the reward untouched")

    # ---- Test 3: zero divergence (logp == ref) => reward unchanged, ALL estimators ----
    lp = torch.randn(2, 4)
    for est in ["kl1", "kl2", "kl3"]:
        o = apply_reward_kl(total, lp, lp.clone(), mask, beta, est)
        assert torch.allclose(o, total, atol=1e-6), (est, o)
    print("✓ 3: zero KL when policy == reference, all estimators")

    # ---- Test 4: k1 exact.  active lr=[0.2,-0.1,0.5]; mean(-lr)=-0.2; 1.0 - 0.2*(-0.2)=1.04 ----
    out = apply_reward_kl(total, logp, ref, mask, beta, "kl1")
    assert torch.allclose(out[0,0], torch.tensor(1.04), atol=1e-5), out[0,0]
    assert torch.allclose(out[1,0], torch.tensor(2.0),  atol=1e-6), out[1,0]
    print("✓ 4: k1 exact")

    # ---- Test 5: k2 exact.  lr^2/2 mean = (0.02+0.005+0.125)/3 = 0.05; 1.0 - 0.2*0.05 = 0.99 ----
    out = apply_reward_kl(total, logp, ref, mask, beta, "kl2")
    assert torch.allclose(out[0,0], torch.tensor(0.99), atol=1e-5), out[0,0]
    print("✓ 5: k2 exact")

    # ---- Test 6: k3 exact AND finite despite padding overflow (mask-before-exp) ----
    # k3 mean over row0 active ~= 0.0583205; 1.0 - 0.2*that = 0.9883359
    out = apply_reward_kl(total, logp, ref, mask, beta, "kl3")
    assert out.isfinite().all(), out
    assert torch.allclose(out[0,0], torch.tensor(0.9883359), atol=1e-5), out[0,0]
    assert torch.allclose(out[1,0], torch.tensor(2.0),      atol=1e-6), out[1,0]
    print("✓ 6: k3 exact, and finite despite unmasked-overflow trap")

    print("\nAll apply_reward_kl tests passed.")

_test_apply_reward_kl()

✓ 1: shape and dtype
✓ 2: beta=0 leaves the reward untouched
✓ 3: zero KL when policy == reference, all estimators
✓ 4: k1 exact
✓ 5: k2 exact
✓ 6: k3 exact, and finite despite unmasked-overflow trap

All apply_reward_kl tests passed.


In [10]:
def ppo_loss(log_probs, log_probs_old, advantages, values, values_old, action_mask,
             clip_eps_lo, clip_eps_hi, clip_eps_val, vf_coef):
    # log_probs, values : (B, S) CURRENT policy logps and CURRENT value estimates (differentiable)
    # log_probs_old, values_old, advantages : (B, S) frozen rollout quantities (constants)
    #
    # returns  = advantages + values_old                    # rebuild returns here, not in GAE
    # value loss (clipped, PESSIMISTIC = max of the two squared errors):
    #   values_clipped = clamp(values, values_old - clip_eps_val, values_old + clip_eps_val)
    #   val_loss = max( 0.5*(returns-values)^2 , 0.5*(returns-values_clipped)^2 )
    # policy loss (your clipped-min surrogate, per token):
    #   ratio = exp(log_probs - log_probs_old)
    #   policy_loss = -min( ratio*A , clamp(ratio, 1-lo, 1+hi)*A )
    # combine + reduce:
    #   loss = policy_loss + vf_coef * val_loss
    #   loss = masked_mean(loss, action_mask, dim=-1).mean(dim=0)   # per-seq, THEN batch
    ...
    diff_logps = torch.exp(log_probs - log_probs_old.detach())
    loss1 = - torch.minimum(diff_logps * advantages.detach(), torch.clamp(diff_logps, min=1 - clip_eps_lo, max=1 + clip_eps_hi) * advantages.detach())
    values_clip = torch.clamp(values, values_old.detach() - clip_eps_val, values_old.detach() + clip_eps_val)
    g = values_old + advantages
    loss2 = torch.maximum(0.5 * (g - values_clip)**2, 0.5 * (g - values)**2)
    loss = loss1 + vf_coef * loss2
    n = torch.sum(action_mask, dim=-1, keepdim=True).clamp(1.0)
    loss = torch.mean(torch.sum(loss * action_mask, dim=-1, keepdim=True) / n)
    return loss


In [11]:
import torch
import math

def _test_ppo_loss():
    mask  = torch.tensor([[1.,1.,1.,0.],[1.,1.,1.,1.]])   # ragged: row0 len3, row1 len4
    A     = torch.tensor([[1.0, 2.0, -1.0, 0.0],[0.5,0.5,0.5,0.5]])
    zeros = torch.zeros(2,4)

    # ---- Test 1: returns a finite scalar ----
    L = ppo_loss(zeros, zeros, A, zeros, zeros, mask, 0.2, 0.2, 0.2, 0.5)
    assert L.dim() == 0, L.shape
    assert L.isfinite()
    print("✓ 1: scalar output")

    # ---- Test 2: anchor. ratio=1 AND values=values_old =>
    #      per-token loss = -A + vf_coef*0.5*A^2, reduced per-seq then batch. ----
    L = ppo_loss(zeros, zeros, A, zeros, zeros, mask, 0.2, 0.2, 0.2, vf_coef=0.5)
    pt = -A + 0.25*A**2                                   # 0.5*0.5 = 0.25
    expected = ((pt[0,:3].mean()) + (pt[1,:4].mean())) / 2
    assert torch.allclose(L, expected, atol=1e-6), (L, expected)
    assert abs(L.item() - (-0.3020833)) < 1e-5, L.item()
    print("✓ 2: anchor at ratio=1, values=values_old")

    # ---- Test 3: masking. Corrupt padded A / values / values_old; loss must not move. ----
    Ac, vc, voc = A.clone(), zeros.clone(), zeros.clone()
    Ac[mask==0] += 999.; vc[mask==0] += 999.; voc[mask==0] += 999.
    Lc = ppo_loss(zeros, zeros, Ac, vc, voc, mask, 0.2, 0.2, 0.2, vf_coef=0.5)
    assert torch.allclose(Lc, L, atol=1e-6), (Lc, L)
    print("✓ 3: padding excluded from the loss")

    # ---- Test 4: value clipping is pessimistic (MAX). Current value jumped 9 past
    #      V_old=0 with returns=10; clipped error must win over the small unclipped one. ----
    m1 = torch.tensor([[1.]])
    L = ppo_loss(torch.zeros(1,1), torch.zeros(1,1),
                 torch.tensor([[10.]]), torch.tensor([[9.]]), torch.tensor([[0.]]),
                 m1, 0.2, 0.2, clip_eps_val=0.2, vf_coef=0.5)
    # policy=-10 ; clipped val = 0.5*(10-0.2)^2 = 48.02 (>> unclipped 0.5) ; -10 + 0.5*48.02
    assert torch.allclose(L, torch.tensor(14.01), atol=1e-3), L
    assert abs(L.item() - (-9.75)) > 1.0                 # NOT the unclipped-only answer
    print("✓ 4: value loss takes the clipped (pessimistic) branch")

    # ---- Test 5: policy clip caps the upside. A>0, ratio=1.5 -> clipped to 1.2. ----
    L = ppo_loss(torch.tensor([[math.log(1.5)]]), torch.zeros(1,1),
                 torch.tensor([[1.]]), torch.zeros(1,1), torch.zeros(1,1),
                 m1, 0.2, 0.2, 0.2, vf_coef=0.0)
    assert torch.allclose(L, torch.tensor(-1.2), atol=1e-5), L   # uses 1.2, not 1.5
    print("✓ 5: policy clip caps the surrogate at the band edge")

    # ---- Test 6: reduction is per-sequence-then-batch, NOT a global token mean. ----
    mR = torch.tensor([[1.,0.,0.,0.],[1.,1.,1.,1.]])         # lengths 1 and 4
    AR = torch.tensor([[4.,0.,0.,0.],[1.,1.,1.,1.]])
    L = ppo_loss(torch.zeros(2,4), torch.zeros(2,4), AR,
                 torch.zeros(2,4), torch.zeros(2,4), mR, 0.2, 0.2, 0.2, vf_coef=0.0)
    # per-seq: seq0 mean(-4)=-4 ; seq1 mean(-1)=-1 ; batch mean = -2.5
    # a GLOBAL token mean would give (-4-1-1-1-1)/5 = -1.6, which must NOT match
    assert torch.allclose(L, torch.tensor(-2.5), atol=1e-6), L
    assert abs(L.item() - (-1.6)) > 1e-3
    print("✓ 6: per-sequence-then-batch reduction (short seqs weighted equally)")

    print("\nAll ppo_loss tests passed.")

_test_ppo_loss()

✓ 1: scalar output
✓ 2: anchor at ratio=1, values=values_old
✓ 3: padding excluded from the loss
✓ 4: value loss takes the clipped (pessimistic) branch
✓ 5: policy clip caps the surrogate at the band edge
✓ 6: per-sequence-then-batch reduction (short seqs weighted equally)

All ppo_loss tests passed.


In [25]:
def compute_log_probs(model, sequence_ids, attention_mask):
    # model(input_ids=..., attention_mask=..., use_cache=False) -> output with .logits (B, S, V)
    #
    # logits    = output.logits[:, :-1, :]      # positions 0..S-2 predict tokens 1..S-1
    # log_probs = log_softmax(logits, dim=-1)    # log_softmax, NOT softmax-then-log
    # targets   = sequence_ids[:, 1:]            # the tokens actually produced
    # gather target log-probs along vocab -> (B, S-1)
    ...
    output = model(input_ids=sequence_ids, attention_mask=attention_mask, use_cache=False)
    logits = output.logits[:,:-1,:]
    log_probs = torch.log_softmax(logits, dim=-1)
    targets = sequence_ids[:, 1:]
    return torch.gather(log_probs, dim=-1, index=targets.unsqueeze(2)).squeeze()


In [26]:
import torch, torch.nn.functional as F, math

class _Out:
    def __init__(self, logits): self.logits = logits
class StubModel:
    """Stand-in for an HF causal LM: returns fixed logits (B,S,V) with a .logits attr."""
    def __init__(self, logits): self._logits = logits
    def __call__(self, input_ids, attention_mask=None, use_cache=False):
        return _Out(self._logits)

def _test_compute_log_probs():
    V = 5

    # ---- Test 1: shape (B,S) -> (B,S-1), dtype float32 ----
    m = StubModel(torch.randn(2, 4, V))
    out = compute_log_probs(m, torch.randint(0, V, (2,4)), torch.ones(2,4))
    assert out.shape == (2, 3), out.shape
    assert out.dtype == torch.float32
    print("✓ 1: shape (B,S)->(B,S-1) and dtype")

    # ---- Test 2: uniform logits => every logp is exactly -log(V) ----
    out = compute_log_probs(StubModel(torch.zeros(1,3,V)), torch.tensor([[0,1,2]]), torch.ones(1,3))
    assert torch.allclose(out, torch.full((1,2), -math.log(V)), atol=1e-5), out
    print("✓ 2: uniform logits give -log(V)")

    # ---- Test 3: THE SHIFT. Position p peaked on token (p+1). Targets that match the
    #      peaks give logp~0; the same logits with targets that miss give logp~-10.
    #      This fails if position p is scored against token p instead of token p+1. ----
    logits = torch.zeros(2, 4, V)
    for p, tok in enumerate([1,2,3]):
        logits[:, p, tok] = 10.0
    seq = torch.tensor([[0,1,2,3],     # targets 1,2,3 match peaks
                        [0,4,4,4]])     # targets 4,4,4 miss peaks
    out = compute_log_probs(StubModel(logits), seq, torch.ones(2,4))
    assert out[0].abs().max() < 1e-3, out[0]              # matched -> ~0
    assert torch.allclose(out[1], torch.full((3,), -10.0), atol=1e-3), out[1]   # missed -> ~-10
    print("✓ 3: position p scored against token p+1 (teacher-forcing shift)")

    # ---- Test 4: hand-computed exact value. logits[pos0]=[0, ln3], target token 1
    #      => log_softmax picks ln3 - log(1+3) = ln(3/4). ----
    lg = torch.tensor([[[0.0, math.log(3)], [0.0, 0.0]]])   # (1,2,2)
    out = compute_log_probs(StubModel(lg), torch.tensor([[0,1]]), torch.ones(1,2))
    assert torch.allclose(out, torch.tensor([[math.log(3/4)]]), atol=1e-5), out
    print("✓ 4: matches hand-computed log_softmax value")

    # ---- Test 5: gathers the RIGHT token. Swap the target and the logp must track it. ----
    lg = torch.tensor([[[math.log(1), math.log(9)], [0.,0.]]])   # pos0: p(tok0)=0.1, p(tok1)=0.9
    out0 = compute_log_probs(StubModel(lg), torch.tensor([[5, 0]]), torch.ones(1,2))  # target tok0
    out1 = compute_log_probs(StubModel(lg), torch.tensor([[5, 1]]), torch.ones(1,2))  # target tok1
    assert torch.allclose(out0, torch.tensor([[math.log(0.1)]]), atol=1e-5), out0
    assert torch.allclose(out1, torch.tensor([[math.log(0.9)]]), atol=1e-5), out1
    print("✓ 5: gather indexes the target token, not a fixed slot")

    print("\nAll compute_log_probs tests passed.")

_test_compute_log_probs()

✓ 1: shape (B,S)->(B,S-1) and dtype
✓ 2: uniform logits give -log(V)
✓ 3: position p scored against token p+1 (teacher-forcing shift)
✓ 4: matches hand-computed log_softmax value
✓ 5: gather indexes the target token, not a fixed slot

All compute_log_probs tests passed.
